# Sequential E2T2 on the peg-slide: one sequence as a GIF

The method is `Sequential_E2T2_peg_slide_2D.ipynb`, loaded below. Set the sequence in the next cell and run all; the GIF goes to `final/output/`.

Each frame shows the peg with its tilt, the block and the barrier where they are at that step, the trial's path so far, and the datapoints sized by their weight in the target, so the sliding target's window and its retreat during a stall can be watched. Between trials only the state and the block are reset; the statistic is handed on.

In [1]:
%%capture method
%run -i Sequential_E2T2_peg_slide_2D.ipynb

In [2]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
from IPython import get_ipython
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Polygon

m = SimpleNamespace(**get_ipython().user_ns)  # the method notebook's names

METHOD = "sliding + forgetting + retry"  # a key of METHODS
OFFSET, SEQ, N_TRIALS = -0.096, 1, 5  # SEQ seeds the noise, as in the experiments
STRIDE, FPS = 20, 25  # steps per frame, frames per second
PEG_LENGTH = 0.3  # drawn only
OUT = Path("../../output")

In [3]:
sliding, lam, retry = m.METHODS[METHOD]
trials = m.run_sequence(sliding, lam, retry, OFFSET, SEQ, N_TRIALS)
for c, r in enumerate(trials):
    outcome = "goal" if r["success"] else "slip-out" if r["inserted"] else "never inserted"
    print(f"trial {c + 1}: {outcome} in {r['steps']} steps, retrying {r['retry_share']:.2f}")

trial 1: goal in 2362 steps, retrying 0.19
trial 2: goal in 841 steps, retrying 0.11
trial 3: goal in 3083 steps, retrying 0.11
trial 4: goal in 4293 steps, retrying 0.26
trial 5: goal in 1319 steps, retrying 0.17


In [4]:
# (trial, state) per frame; each trial's last state is held for a second
frames = [
    (c, k)
    for c, r in enumerate(trials)
    for k in [*range(1, len(r["path"]), STRIDE), *[len(r["path"]) - 1] * FPS]
]


def peg(s):  # the peg's outline, tip at s[:2], tilted by s[2] / DEPTH from vertical
    t = s[2] / m.DEPTH
    up, side = np.array([np.sin(t), np.cos(t)]), m.PEG * np.array([np.cos(t), -np.sin(t)])
    return [
        s[:2] - side,
        s[:2] + side,
        s[:2] + side + PEG_LENGTH * up,
        s[:2] - side + PEG_LENGTH * up,
    ]


fig, ax = plt.subplots(figsize=(6, 6.4))


def draw(frame):
    c, k = frame
    r = trials[c]
    phi, rho = r["phi"][k - 1], r["rho"][k - 1]  # what the law saw before stepping to state k
    world = m.Slide(OFFSET, c=r["c"][k], barrier=r["barrier"][k])
    ax.clear()
    m.draw_world(ax, world)
    w = m.weights(phi, rho) if sliding else np.full(m.N_P, 1 / m.N_P)
    ax.scatter(*m.X_p[:, :2].T, s=40 * m.N_P * w, c="tab:blue", alpha=0.5)
    ax.plot(*r["path"][: k + 1, :2].T, color="0.5", lw=0.5)
    ax.add_patch(Polygon(peg(r["path"][k]), color="tab:red"))
    ax.set_xticks([]), ax.set_yticks([])
    end = ""
    if k == len(r["path"]) - 1:
        end = ": goal" if r["success"] else ": slip-out" if r["inserted"] else ": never inserted"
    ax.set_title(
        f"{METHOD}, offset {OFFSET:+.3f}\n"
        f"trial {c + 1}/{N_TRIALS}, step {k}{end}\nphase {phi:.2f}, retreat {rho / m.delta:.1f} stages",
        fontsize=9,
    )


name = f"sequence_{METHOD.replace(' + ', '_').replace(' ', '_')}_{OFFSET:+.3f}_seq{SEQ}.gif"
FuncAnimation(fig, draw, frames=frames).save(OUT / name, writer=PillowWriter(fps=FPS), dpi=60)
plt.close(fig)
print(f"{len(frames)} frames to {(OUT / name).resolve()}")

723 frames to /home/jens/workspace/docker_intern_PiPER_ergodic/final/output/sequence_sliding_forgetting_retry_-0.096_seq1.gif
